# 🧪**Agent Response Evaluation with - Llumo AI**

---

📘 **Notebook Overview**

This notebook demonstrates how to evaluate agent-generated responses using the **LlumoClient API**. It includes:

- Loading structured data with required keys
- Evaluating agent response using `evaluateAgentResponses()`
- Viewing the evaluation results dataframe

This setup is ideal for internal QA testing, benchmarking agent responses, or auditing performance of function-calling agents in conversational systems.

---


# **⬇ Step 1: Necessary Installation**

In [13]:
!pip install llumo -q

# 📦 **Step 2: Import Required Libraries**


In [14]:
import pandas as pd

# 🔐 **Step 3: Load Llumo API Key**


In [15]:
import os

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "key_MThiM2E0NDM0NDQ3YThlZGMzOTQ3OTU0_46c2f578d3c35625f06f02524dbf38822638bd335c74e05257267df56f9b37a81e18a1a879d6738dc9dc55f7768a4a53aa19cfe06fa90e91918211cfdd60ce731a6a6fc924f5273c2cdf8ca2befa1c40f345e8d56ad7cbcdac135101f32c8cdfcaddb1e0d377e9250a9ea08924cbe283c4c405f1de42067e88ba3da74cdb00fe"

llumo_key = os.getenv("LLUMO_API_KEY")

# 📁 **Step 4: Load Excel Data - Optional**


📌 Note:
The input data for agent evaluation must have following mandatory keys for each query result:

- query

- output

- messageHistory

- tools


```
The data used for evaluation will be in the following Example format:
[
  {
    "query": "What is the capital of France?",
    "output": "The capital of France is Paris.",
    "messageHistory": '''[{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "The capital of France is Paris."}]''',
    "tools": "{'tool_1_Name':'description','tool_2_Name':'description'}"
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families.",
    "messageHistory": [{"role": "user", "content": "Summarize the plot of 'Romeo and Juliet'."}, {"role": "assistant", "content": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."}],
    "tools": "{'tool_1_Name':'description','tool_2_Name':'description'}"
  }
]
```

In [12]:
import pandas as pd
import json

data = [
  {
    "query": "What is on the menu today?",
    "output": "Here is the current menu: ...",
    "groundTruth": "Here’s what’s currently available on the menu.",
    "context": "The restaurant menu includes: Pizza ($12, 10 available), Burger ($8, 15 available), Soda ($2, 30 available). Use getMenu() to retrieve current offerings.",
    "messageHistory": "[{'role': 'user', 'content': 'What is on the menu today?'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_761022bafffb4f53908481dfd6ab1cb8', 'function': {'name': 'getMenu', 'arguments': '{}'}, 'type': 'function'}]}, {'role': 'tool', 'tool_call_id': 'call_761022bafffb4f53908481dfd6ab1cb8', 'content': \"{'message': 'Here is the current menu: ...'}\"}, {'role': 'assistant', 'content': 'Here is the current menu: ...'}]",
    "tools": "{'getMenu': 'Get the restaurant menu.', 'addToCart': 'Add an item to the cart. Requires item name and quantity as parameters.', 'removeFromCart': 'Remove an item from the cart. Requires item name and quantity as parameters.', 'getOrderDetails': 'Get the order details and generate an order ID. Returns order summary and clears the cart.', 'clearCart': 'Clear all items from the cart and restock them in the menu.', 'viewOrderHistory': 'View past order history including items ordered and totals.'}"
  },
  {
    "query": "Add 2 pizzas to my cart",
    "output": "Added 2 pizzas to your cart.",
    "groundTruth": "2 pizzas have been successfully added to your cart.",
    "context": "Pizza is available for $12 each (current stock: 10). To add items, use addToCart(item, quantity) with the item name and desired quantity.",
    "messageHistory": "[{'role': 'user', 'content': 'Add 2 pizzas to my cart'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_897cc28537994cf3b6e34ec576246f71', 'function': {'name': 'addToCart', 'arguments': '{\"item\": \"pizza\", \"quantity\": 2}'}, 'type': 'function'}]}, {'role': 'tool', 'tool_call_id': 'call_897cc28537994cf3b6e34ec576246f71', 'content': \"{'message': 'Added 2 pizzas to your cart.'}\"}, {'role': 'assistant', 'content': 'Added 2 pizzas to your cart.'}]",
    "tools": "{'getMenu': 'Get the restaurant menu.', 'addToCart': 'Add an item to the cart. Requires item name and quantity as parameters.', 'removeFromCart': 'Remove an item from the cart. Requires item name and quantity as parameters.', 'getOrderDetails': 'Get the order details and generate an order ID. Returns order summary and clears the cart.', 'clearCart': 'Clear all items from the cart and restock them in the menu.', 'viewOrderHistory': 'View past order history including items ordered and totals.'}"
  },
  {
    "query": "Remove 1 burger from my cart",
    "output": "Removed 1 burger from your cart.",
    "groundTruth": "1 burger has been removed from your cart.",
    "context": "Burger is $8 each (current stock: 15). Use removeFromCart(item, quantity) to reduce items in cart. Ensure item exists in cart first.",
    "messageHistory": "[{'role': 'user', 'content': 'Remove 1 burger from my cart'}, {'role': 'assistant', 'tool_calls': [{'id': 'call_333e2b59671b4e0f9cc4534a6c165403', 'function': {'name': 'removeFromCart', 'arguments': '{\"item\": \"burger\", \"quantity\": 1}'}, 'type': 'function'}]}, {'role': 'tool', 'tool_call_id': 'call_333e2b59671b4e0f9cc4534a6c165403', 'content': \"{'message': 'Removed 1 burger from your cart.'}\"}, {'role': 'assistant', 'content': 'Removed 1 burger from your cart.'}]",
    "tools": "{'getMenu': 'Get the restaurant menu.', 'addToCart': 'Add an item to the cart. Requires item name and quantity as parameters.', 'removeFromCart': 'Remove an item from the cart. Requires item name and quantity as parameters.', 'getOrderDetails': 'Get the order details and generate an order ID. Returns order summary and clears the cart.', 'clearCart': 'Clear all items from the cart and restock them in the menu.', 'viewOrderHistory': 'View past order history including items ordered and totals.'}"
  }
]

# 🧠 **Step 5: Evaluate Agent Responses using LlumoClient**


In [16]:
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key=llumo_key)

# Evaluate agent responses based on queries and outputs
resultdf = client.evaluateAgentResponses(
    data = data, # input data
    evals = ['Tool Reliability','Stepwise Progression','Tool Selection Accuracy','Final Task Alignment'], # tool eval metrics
    getDataFrame=True,  # Return result as a DataFrame (True) or dictionary (False)
    createExperiment=False  # When True, creates an experiment (no result object returned here)

)


Processing Batches: 100%|██████████| 4/4 [00:14<00:00,  3.70s/batch]


# 📊 **Step 6: View Evaluation Results**


In [17]:
# Print the results dataframe - first 5 records
resultdf.head()


,query,output,groundTruth,context,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,What is on the menu today?,Here is the current menu: ...,Here’s what’s currently available on the menu.,"The restaurant menu includes: Pizza ($12, 10 a...","[{'role': 'user', 'content': 'What is on the m...","{'getMenu': 'Get the restaurant menu.', 'addTo...",99,The `getMenu` tool successfully executed and r...,99,The user query is 'What is on the menu today?'...,99,The user asked for the menu. The assistant us...,100,The assistant successfully retrieved and prese...
1,Add 2 pizzas to my cart,Added 2 pizzas to your cart.,2 pizzas have been successfully added to your ...,Pizza is available for $12 each (current stock...,"[{'role': 'user', 'content': 'Add 2 pizzas to ...","{'getMenu': 'Get the restaurant menu.', 'addTo...",100,The `addToCart` tool successfully added pizzas...,99,The tool 'addToCart' is relevant to the user q...,99,"The assistant used only the `addToCart` tool, ...",99,The assistant successfully added two pizzas to...
2,Remove 1 burger from my cart,Removed 1 burger from your cart.,1 burger has been removed from your cart.,Burger is $8 each (current stock: 15). Use rem...,"[{'role': 'user', 'content': 'Remove 1 burger ...","{'getMenu': 'Get the restaurant menu.', 'addTo...",99,The tool successfully removed the burger from ...,100,The tool 'removeFromCart' is relevant to the u...,99,The assistant used only the 'removeFromCart' t...,99,The assistant successfully removed the burger ...
